Migrated from Databricks to Neuron (auto-port).

In [ ]:
import os
import sys
from pathlib import Path

for _candidate in (
    "/home/jovyan/work/default/rashmi-ranjan/lib",
    str(Path.cwd().parent / "lib"),
    str(Path.cwd() / "lib"),
):
    if _candidate not in sys.path and Path(_candidate).is_dir():
        sys.path.insert(0, _candidate)

from neuron_runtime import (
    GoogleSheetReader,
    SlackReportsTrigger,
    get_sql_df,
    load_service_account,
    load_slack_token,
    make_unique_columns,
)

service_account = load_service_account()
slack_token = load_slack_token()

In [ ]:
from datetime import datetime, timedelta

now = datetime.now() + timedelta(hours=5,minutes=30)

if(now.weekday() == 6):
    import sys
    sys.exit("Today is Sunday, no need to run the notebook")

In [ ]:
import os
import shutil

# Define the Google Sheet URL and worksheet name
gsheet_url = "https://docs.google.com/spreadsheets/d/1pV1Rx7fkuJagQRulyPAI9GCdJ5NbOYi7mc94XNR8E28/edit?gid=0#gid=0"
sheet_name = "Slack Card"

# Define the destination folder for the image
destination_folder = "data"

# Create the data directory if it doesn't exist
if not os.path.exists(destination_folder):
    os.makedirs(destination_folder)

# Initialize GoogleSheetReader with the provided sheet link and service account credentials
google_sheet_reader = GoogleSheetReader(gsheet_url, service_account)

# Download the specified worksheet as an image (direct screenshot)
image_path = google_sheet_reader.download_google_sheet_as_image(
    destination_folder,
    sheet_name
)

# Initialize SlackReportsTrigger with the Slack token
slack_channel_ids = [
    "C0B77EWGSRE",  # Channel where users should be tagged
    "C08CXD3QVLH",  # Channel where users should NOT be tagged
]

slack = SlackReportsTrigger(slack_token)

# Base message
base_message = "Please find below the summary of Motor Sales advisors audits done as on today"

# Send the image to both Slack channels
for slack_channel_id in slack_channel_ids:

    # Add mentions only for the specified channel
    if slack_channel_id == "C0B77EWGSRE":
        message = (
            f"{base_message}\n"
            "cc: <@U0ATV6LBBC5>"
        )
    else:
        message = base_message

    slack.send_file_to_slack(
        slack_channel_id,
        image_path,
        title="Motor Sales Audit Summary Snapshot",
        message=message
    )